# Διάλεξη 3 — RAG: Chatbot στα Δικά σου Δεδομένα

**Μάθημα:** Εισαγωγή στην Τεχνητή Νοημοσύνη  
**Lab:** Retrieval-Augmented Generation · ChromaDB · sentence-transformers · Ollama

---

## Πώς δουλεύει αυτό το notebook

Άλλαξε **μόνο** το **Knobs cell** παρακάτω. Όλα τα υπόλοιπα cells τρέχουν αυτόματα.

| Section | Τι κάνουμε |
|---------|------------|
| 1. Setup | Εγκατάσταση, imports, δημιουργία corpus |
| 2. Vector Store | Embeddings + ChromaDB index |
| 3. Semantic Search | Query → retrieve relevant chunks |
| 4. RAG Answer | Prompt + LLM → απάντηση με πηγές |
| 5. Swap Your Data | Template για δικά σου δεδομένα |
| 6. Αξιολόγηση | Retrieval quality check |

> **Tip:** Τρέξε πρώτα με τα EuroLeague data, μετά αντικατέστησε με δικά σου docs στο Section 5!

In [1]:
# ==========================================================
# KNOBS — Αλλαξε μονο εδω
# ==========================================================

DATASET           = "euroleague"
EUROLEAGUE_PATH   = "euroleague.csv"
EUROLEAGUE_SEASON = "E2024"

N_RESULTS         = 5
EMBED_MODEL       = "all-MiniLM-L6-v2"
COLLECTION_NAME   = "euroleague_rag"

USE_OLLAMA        = True
OLLAMA_MODEL      = "llama3.2"

USE_OPENAI        = False
OPENAI_API_KEY    = ""
OPENAI_MODEL      = "gpt-4o-mini"

print('Knobs loaded')
print(f'  Dataset: {DATASET} | Embed: {EMBED_MODEL} | top-k: {N_RESULTS}')
print(f'  LLM: {"Ollama" if USE_OLLAMA else "OpenAI" if USE_OPENAI else "Template (no LLM)"}')

Knobs loaded
  Dataset: euroleague | Embed: all-MiniLM-L6-v2 | top-k: 5
  LLM: Ollama


---
## 1. Setup & Corpus

### Τι χρειαζομαστε

- **sentence-transformers**: δημιουργεί embeddings τοπικά (δωρεάν, χωρίς API)
- **chromadb**: vector database
- Και τα δύο τρέχουν **100% τοπικά** — δεν χρειάζεσαι API key για τα Sections 1-3!

> 💡 Αν δεν έχεις τα packages: `!pip install sentence-transformers chromadb pandas numpy`

In [2]:
# Εγκατασταση (uncomment αν χρειαζεται):
# !pip install sentence-transformers chromadb pandas numpy --quiet

import numpy as np
import pandas as pd
import json, os, time

from sentence_transformers import SentenceTransformer
import chromadb

import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

/home/kostas/Machine_Learning_Projects/lab_3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


### Δημιουργια Corpus

Το **corpus** ειναι η συλλογη κειμενων που θα ψαξει το RAG.  
Καθε γραμμη = ενα **chunk** (μικρο κομματι κειμενου).

Για το EuroLeague demo, καθε chunk = στατιστικα ενος παικτη σε μια σεζον.

> 💡 Το Section 5 δειχνει πως να αντικαταστησεις με δικα σου docs!

In [3]:
def build_euroleague_corpus(path, season=None):
    """Φορτωση EuroLeague δεδομενων και δημιουργια text corpus."""
    df = pd.read_csv(path)
    if season:
        df = df[df["season_code"] == season].copy()

    stat_cols = [
        "valuation_per_game", "points_per_game", "assists_per_game",
        "total_rebounds_per_game", "steals_per_game", "minutes_per_game",
        "three_points_percentage", "two_points_percentage", "games_played"
    ]
    stat_cols = [c for c in stat_cols if c in df.columns]

    corpus, ids, metas = [], [], []
    stat_map = {
        "points_per_game": "pts/g", "assists_per_game": "ast/g",
        "total_rebounds_per_game": "reb/g", "steals_per_game": "stl/g",
        "minutes_per_game": "min/g", "valuation_per_game": "val/g",
        "three_points_percentage": "3P%", "two_points_percentage": "2P%",
        "games_played": "GP",
    }
    for idx, (_, row) in enumerate(df.iterrows()):
        name = str(row.get("player_name", row.get("player_id", "Unknown")))
        team = str(row.get("team_name", row.get("team_id", "")))
        szn  = str(row.get("season_code", ""))
        stats_str = ", ".join(
            f"{stat_map.get(col, col)}={row[col]:.1f}"
            for col in stat_cols if pd.notna(row.get(col))
        )
        # Starter role
        role = ""
        if "games_started" in df.columns and "games_played" in df.columns:
            gp = row.get("games_played", 0)
            gs = row.get("games_started", 0)
            if gp > 0:
                role = f" | Role: {'starter' if gs/gp > 0.5 else 'bench'}"
        text = f"{name} ({team}, {szn}): {stats_str}{role}"
        corpus.append(text)
        # Use idx to guarantee unique IDs even if a player appears in multiple rows
        ids.append(f"{name}_{szn}_{idx}".replace(" ", "_")[:60])
        metas.append({"player": name, "team": team, "season": szn})
    return corpus, ids, metas


# Φορτωση
if DATASET == "euroleague":
    try:
        corpus, doc_ids, metadatas = build_euroleague_corpus(EUROLEAGUE_PATH, EUROLEAGUE_SEASON)
        print(f"EuroLeague corpus: {len(corpus):,} entries")
        print(f"Παραδειγμα: {corpus[0]}")
    except FileNotFoundError:
        print("EuroLeague CSV δεν βρεθηκε — χρησιμοποιω demo corpus...")
        corpus = [
            "Vasilis Sloukas (Olympiacos, E2024): pts/g=14.2, ast/g=8.1, reb/g=3.2, val/g=18.5, GP=30 | Role: starter",
            "Nikola Milutinov (Olympiacos, E2024): pts/g=12.1, ast/g=1.2, reb/g=8.5, val/g=16.8, GP=29 | Role: starter (center)",
            "Shane Larkin (Anadolu Efes, E2022): pts/g=19.2, ast/g=5.4, reb/g=3.1, val/g=22.1, GP=32 | Role: starter (MVP season)",
            "Mike James (AS Monaco, E2024): pts/g=20.5, ast/g=6.2, reb/g=3.8, val/g=24.3, GP=31 | Role: starter (top scorer)",
            "Jan Vesely (Fenerbahce, E2023): pts/g=9.4, ast/g=1.8, reb/g=7.2, val/g=14.1, GP=27 | Role: starter (center)",
            "Sergio Rodriguez (Olympiacos, E2022): pts/g=10.5, ast/g=6.8, reb/g=2.1, val/g=14.3, GP=25 | Role: starter (playmaker)",
            "Nando De Colo (CSKA, E2019): pts/g=16.8, ast/g=4.2, reb/g=3.5, val/g=19.2, GP=30 | Role: starter",
            "Nikola Kalinic (Barcelona, E2021): pts/g=13.2, ast/g=3.8, reb/g=4.1, val/g=16.5, GP=28 | Role: starter",
            "Aleksej Pokusevski (Olympiacos, E2024): pts/g=8.3, ast/g=2.1, reb/g=5.8, val/g=11.2, GP=20 | Role: bench",
            "Kostas Sloukas (Fenerbahce, E2023): pts/g=13.8, ast/g=7.9, reb/g=2.8, val/g=17.2, GP=28 | Role: starter (PG)",
        ]
        doc_ids   = [f"doc_{i}" for i in range(len(corpus))]
        metadatas = [{"player": f"Player {i}"} for i in range(len(corpus))]
        print(f"Demo corpus: {len(corpus)} entries")
elif DATASET == "custom":
    print("Custom dataset — τρεξε το Section 5!")
    corpus, doc_ids, metadatas = [], [], []

EuroLeague corpus: 312 entries
Παραδειγμα: P003733 (MAD, E2024): val/g=6.6, pts/g=5.9, ast/g=1.7, reb/g=2.4, stl/g=0.5, min/g=18.9, 3P%=0.5, 2P%=0.6, GP=39.0 | Role: starter


---
## 2. Build Vector Store

### Τι κανουμε εδω

1. **Φορτωνουμε το embedding model** — μετατρεπει κειμενο σε αριθμους
2. **Δημιουργουμε ChromaDB collection** — η μνημη του chatbot
3. **Προσθετουμε τα chunks** — καθε chunk γινεται vector

### Τι θα δεις
- Αριθμος chunks στο store
- Διασταση embedding vector (384 για MiniLM)
- Παραδειγμα: τι μοιαζουν οι αριθμοι ενος chunk

> ⏱️ Η πρωτη φορα παιρνει λιγα λεπτα (download model ~90MB). Μετα γινεται cache.

In [4]:
print(f'Φορτωση embedding model: {EMBED_MODEL}')
t0 = time.time()
embedder = SentenceTransformer(EMBED_MODEL)
print(f'  Model loaded ({time.time()-t0:.1f}s)')

# ChromaDB in-memory client
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except:
    pass
collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)
print(f'ChromaDB collection ready')

if corpus:
    print(f'Embedding {len(corpus)} chunks...')
    t1 = time.time()
    embeddings = embedder.encode(corpus, show_progress_bar=True, batch_size=32)
    collection.add(
        documents=corpus,
        embeddings=embeddings.tolist(),
        ids=doc_ids,
        metadatas=metadatas,
    )
    print(f'Vector store ready: {collection.count()} chunks ({time.time()-t1:.1f}s)')
    print(f'Embedding dim: {embeddings.shape[1]}')
    print(f'Παραδειγμα (πρωτοι 8 αριθμοι): [{chr(44)+chr(32).join(f"{v:.3f}" for v in embeddings[0][:8])}, ...]')
    print(f'-> Αυτοι οι {embeddings.shape[1]} αριθμοι αναπαριστουν το νοημα του κειμενου!')
else:
    print('Corpus κενο — δες Section 5.')

Φορτωση embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 611.31it/s]


  Model loaded (8.0s)
ChromaDB collection ready
Embedding 312 chunks...


Batches: 100%|██████████| 10/10 [00:03<00:00,  3.06it/s]


Vector store ready: 312 chunks (3.9s)
Embedding dim: 384
Παραδειγμα (πρωτοι 8 αριθμοι): [,-0.067 0.007 -0.069 -0.019 -0.074 0.052 0.011 0.095, ...]
-> Αυτοι οι 384 αριθμοι αναπαριστουν το νοημα του κειμενου!


---
## 3. Semantic Search — Ψαχνουμε με Νοημα

### Τι κανουμε εδω

Δοκιμαζουμε το **retrieval**: δινουμε μια ερωτηση και βλεπουμε ποια chunks επιστρεφει.

### Τι θα δεις
- Top-k chunks για καθε ερωτηση
- **Cosine similarity** score (1.0 = τελεια αντιστοιχια, 0.0 = αδιαφορο)
- Semantic search βρισκει σχετικο περιεχομενο ακομα και χωρις ακριβεις λεξεις!

> 💡 Δοκιμασε ερωτησεις που **δεν** χρησιμοποιουν ονοματα — π.χ. 'καλος playmaker'

In [5]:
def semantic_search(question, n=None):
    """Semantic search στο vector store."""
    n = n or N_RESULTS
    q_emb = embedder.encode([question]).tolist()
    results = collection.query(
        query_embeddings=q_emb,
        n_results=min(n, collection.count()),
        include=["documents", "distances", "metadatas"]
    )
    docs  = results["documents"][0]
    dists = results["distances"][0]
    metas = results.get("metadatas", [[{}]*len(docs)])[0]
    sims  = [1 - d/2 for d in dists]   # cosine distance -> similarity
    return docs, sims, metas


def display_results(question, docs, sims, metas, n=None):
    n = n or len(docs)
    print(f"Ερωτηση: '{question}'")
    print(f'  Top-{n} chunks:')
    for i, (doc, sim) in enumerate(zip(docs[:n], sims[:n]), 1):
        print(f'  [{i}] sim={sim:.3f} | {doc[:110]}{"..." if len(doc)>110 else ""}')
    print()


# Demo queries
if collection.count() > 0:
    for q in [
        'Ποιος παικτης εχει τα περισσοτερα assists;',
        'Top scorer με υψηλο valuation',
        'Ψηλοι παικτες με πολλα rebounds',
        'Καλος playmaker και περιοπτος οργανωτης',   # semantic — δεν υπαρχει αυτη η φραση
    ]:
        docs_r, sims_r, metas_r = semantic_search(q, n=3)
        display_results(q, docs_r, sims_r, metas_r, n=3)

Ερωτηση: 'Ποιος παικτης εχει τα περισσοτερα assists;'
  Top-3 chunks:
  [1] sim=0.606 | P007870 (PAR, E2024): val/g=10.8, pts/g=9.9, ast/g=1.5, reb/g=3.6, stl/g=0.9, min/g=18.7, 3P%=0.5, 2P%=0.5, GP...
  [2] sim=0.606 | P008962 (ULK, E2024): val/g=18.3, pts/g=16.7, ast/g=1.7, reb/g=5.3, stl/g=1.1, min/g=31.3, 3P%=0.4, 2P%=0.5, G...
  [3] sim=0.603 | P009783 (ULK, E2024): val/g=6.8, pts/g=6.7, ast/g=0.5, reb/g=3.5, stl/g=0.6, min/g=18.4, 3P%=0.4, 2P%=0.5, GP=...

Ερωτηση: 'Top scorer με υψηλο valuation'
  Top-3 chunks:
  [1] sim=0.712 | P001392 (MIL, E2024): val/g=9.9, pts/g=8.3, ast/g=1.7, reb/g=2.8, stl/g=1.1, min/g=21.0, 3P%=0.5, 2P%=0.6, GP=...
  [2] sim=0.711 | P001518 (MIL, E2024): val/g=3.2, pts/g=3.6, ast/g=0.4, reb/g=2.5, stl/g=0.4, min/g=15.1, 3P%=0.4, 2P%=0.7, GP=...
  [3] sim=0.708 | P006509 (MIL, E2024): val/g=4.5, pts/g=3.2, ast/g=1.1, reb/g=1.9, stl/g=0.3, min/g=15.4, 3P%=0.5, 2P%=0.6, GP=...

Ερωτηση: 'Ψηλοι παικτες με πολλα rebounds'
  Top-3 chunks:
  [1] sim=0.594 | P0

### Πειραμα: Αλλαξε την ερωτηση

> Αλλαξε το `MY_QUESTION` παρακατω και τρεξε το cell. Τι chunks βρισκει;

In [6]:
MY_QUESTION = 'Ποιος παικτης παίρνει τα περισσότερα reb/g;'

if collection.count() > 0:
    docs_r, sims_r, metas_r = semantic_search(MY_QUESTION)
    display_results(MY_QUESTION, docs_r, sims_r, metas_r)
    print('Full chunks (αυτα θα δει το LLM):')
    print('-' * 65)
    for i, doc in enumerate(docs_r, 1):
        print(f'  Chunk {i}: {doc}')
        print()

Ερωτηση: 'Ποιος παικτης παίρνει τα περισσότερα reb/g;'
  Top-5 chunks:
  [1] sim=0.634 | P009035 (RED, E2024): val/g=16.6, pts/g=10.2, ast/g=0.6, reb/g=8.3, stl/g=0.9, min/g=24.3, 3P%=0.4, 2P%=0.7, G...
  [2] sim=0.634 | P008173 (BER, E2024): val/g=8.8, pts/g=9.6, ast/g=3.8, reb/g=2.0, stl/g=0.5, min/g=20.7, 3P%=0.3, 2P%=0.4, GP=...
  [3] sim=0.631 | P001133 (VIR, E2024): val/g=17.5, pts/g=15.6, ast/g=2.7, reb/g=5.0, stl/g=0.7, min/g=27.5, 3P%=0.4, 2P%=0.5, G...
  [4] sim=0.630 | P008384 (BER, E2024): val/g=7.7, pts/g=9.7, ast/g=0.8, reb/g=2.2, stl/g=0.7, min/g=20.5, 3P%=0.2, 2P%=0.6, GP=...
  [5] sim=0.628 | P006739 (VIR, E2024): val/g=7.2, pts/g=5.4, ast/g=4.1, reb/g=2.4, stl/g=1.2, min/g=23.6, 3P%=0.3, 2P%=0.5, GP=...

Full chunks (αυτα θα δει το LLM):
-----------------------------------------------------------------
  Chunk 1: P009035 (RED, E2024): val/g=16.6, pts/g=10.2, ast/g=0.6, reb/g=8.3, stl/g=0.9, min/g=24.3, 3P%=0.4, 2P%=0.7, GP=18.0 | Role: starter

  Chunk 2: P008173 (BER

---
## 4. RAG Answer — Απο Ερωτηση σε Απαντηση

### Τι κανουμε εδω

Συνδεουμε ολα τα κομματια:
1. **Retrieve**: παιρνουμε τα top-k chunks
2. **Build prompt**: τα βαζουμε ως context
3. **Generate**: το LLM (η template) παραγει απαντηση

### Τρεις επιλογες LLM
| Επιλογη | Knob | Απαιτηση |
|---------|------|----------|
| **Template** (default) | `USE_OLLAMA=False, USE_OPENAI=False` | Τιποτα |
| **Ollama** (local LLM) | `USE_OLLAMA=True` | Ollama εγκατεστημενο |
| **OpenAI** | `USE_OPENAI=True` | API key |

> 💡 Με `USE_OLLAMA=False` βλεπεις τα retrieved chunks — αρκει για να καταλαβεις το RAG!

In [7]:
def build_rag_prompt(question, chunks):
    context = "\n---\n".join(
        f"[Πηγη {i+1}]: {chunk}"
        for i, chunk in enumerate(chunks)
    )
    system = (
        "Εισαι βοηθος που απανταει ΜΟΝΟ βασει του παρακατω context. "
        "Αν η απαντηση δεν βρισκεται στο context, πες 'Δεν βρηκα αυτη την πληροφορια'. "
        "Αναφερε παντα από ποια πηγη βρηκες την πληροφορια."
    )
    prompt = f"{system}\n\nContext:\n{context}\n\nΕρωτηση: {question}\n\nΑπαντηση:"
    return prompt, context


def generate_answer(question, chunks):
    """Παραγει απαντηση με την επιλεγμενη μεθοδο."""
    prompt, context = build_rag_prompt(question, chunks)

    if USE_OLLAMA:
        try:
            import ollama
            response = ollama.chat(
                model=OLLAMA_MODEL,
                messages=[{"role": "user", "content": prompt}]
            )
            return response["message"]["content"], "ollama"
        except ImportError:
            print("pip install ollama")
            return None, "error"
        except Exception as e:
            print(f"Ollama error: {e} — τρεχει ο server? (ollama serve)")
            return None, "error"

    elif USE_OPENAI:
        try:
            from openai import OpenAI
            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": "Εισαι βοηθος που απανταει βασει δεδομενων."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.3,
            )
            return resp.choices[0].message.content, "openai"
        except Exception as e:
            print(f"OpenAI error: {e}")
            return None, "error"

    else:
        # Template mode — δειχνει τα chunks χωρις LLM
        answer = f"[Template mode — no LLM]\n\nTop-{len(chunks)} retrieved chunks:\n\n"
        for i, chunk in enumerate(chunks, 1):
            answer += f"  [{i}] {chunk}\n\n"
        answer += "-> Με Ollama/OpenAI, αυτα τα chunks θα γινοντουσαν μια συνεκτικη απαντηση."
        return answer, "template"


def rag_query(question):
    """Full RAG pipeline: retrieve -> build prompt -> generate."""
    print(f"\n{'='*65}")
    print(f'Ερωτηση: {question}')
    print(f"{'='*65}")

    print(f'Step 1 — Semantic Search (top-{N_RESULTS})...')
    docs_r, sims_r, _ = semantic_search(question)
    for i, (doc, sim) in enumerate(zip(docs_r, sims_r), 1):
        print(f'  [{i}] sim={sim:.3f} | {doc[:80]}...')

    print(f'Step 2 — RAG prompt: {len(docs_r)} chunks')
    mode = 'Ollama' if USE_OLLAMA else 'OpenAI' if USE_OPENAI else 'Template'
    print(f'Step 3 — Generate ({mode})...')

    answer, mode_used = generate_answer(question, docs_r)
    if answer:
        print(f'\nΑπαντηση [{mode_used}]:')
        print('-' * 65)
        print(answer)
        print('-' * 65)
    return answer


if collection.count() > 0:
    rag_query('Ποιος εχει τις περισσότερες assists ανα παιχνιδι;')


Ερωτηση: Ποιος εχει τις περισσότερες assists ανα παιχνιδι;
Step 1 — Semantic Search (top-5)...
  [1] sim=0.608 | P008962 (ULK, E2024): val/g=18.3, pts/g=16.7, ast/g=1.7, reb/g=5.3, stl/g=1.1, m...
  [2] sim=0.607 | P007870 (PAR, E2024): val/g=10.8, pts/g=9.9, ast/g=1.5, reb/g=3.6, stl/g=0.9, mi...
  [3] sim=0.605 | P008104 (ULK, E2024): val/g=5.8, pts/g=5.9, ast/g=2.3, reb/g=1.1, stl/g=0.4, min...
  [4] sim=0.605 | P007182 (ULK, E2024): val/g=5.2, pts/g=3.5, ast/g=0.6, reb/g=3.7, stl/g=0.4, min...
  [5] sim=0.604 | P009783 (ULK, E2024): val/g=6.8, pts/g=6.7, ast/g=0.5, reb/g=3.5, stl/g=0.6, min...
Step 2 — RAG prompt: 5 chunks
Step 3 — Generate (Ollama)...

Απαντηση [ollama]:
-----------------------------------------------------------------
Δεν βρηκα αυτη την πληροφορια από nào든 πηγη.
-----------------------------------------------------------------


In [8]:
if collection.count() > 0:
    rag_query('Ποιος παικτης ειναι ο καλυτερος scorer;')


Ερωτηση: Ποιος παικτης ειναι ο καλυτερος scorer;
Step 1 — Semantic Search (top-5)...
  [1] sim=0.625 | P007870 (PAR, E2024): val/g=10.8, pts/g=9.9, ast/g=1.5, reb/g=3.6, stl/g=0.9, mi...
  [2] sim=0.617 | P009035 (RED, E2024): val/g=16.6, pts/g=10.2, ast/g=0.6, reb/g=8.3, stl/g=0.9, m...
  [3] sim=0.617 | P011194 (RED, E2024): val/g=16.5, pts/g=14.1, ast/g=1.7, reb/g=4.9, stl/g=0.8, m...
  [4] sim=0.617 | P008232 (PAR, E2024): val/g=4.7, pts/g=3.9, ast/g=1.2, reb/g=2.2, stl/g=0.6, min...
  [5] sim=0.617 | P012609 (PRS, E2024): val/g=7.2, pts/g=7.7, ast/g=1.2, reb/g=2.5, stl/g=0.4, min...
Step 2 — RAG prompt: 5 chunks
Step 3 — Generate (Ollama)...

Απαντηση [ollama]:
-----------------------------------------------------------------
Δεν βρηκα αυτη την πληροφορια από οποιαδήποτε πηγή.
-----------------------------------------------------------------


In [9]:
# Out-of-corpus: τι γινεται αν ρωτησουμε κατι εκτος corpus;
if collection.count() > 0:
    rag_query('Ποια ειναι η καλυτερη πιτσα στην Αθηνα;')


Ερωτηση: Ποια ειναι η καλυτερη πιτσα στην Αθηνα;
Step 1 — Semantic Search (top-5)...
  [1] sim=0.564 | P013383 (PAN, E2024): val/g=6.8, pts/g=6.0, ast/g=0.5, reb/g=3.6, stl/g=0.7, min...
  [2] sim=0.564 | P008173 (BER, E2024): val/g=8.8, pts/g=9.6, ast/g=3.8, reb/g=2.0, stl/g=0.5, min...
  [3] sim=0.564 | P013383 (TEL, E2024): val/g=7.4, pts/g=5.8, ast/g=1.3, reb/g=4.7, stl/g=0.3, min...
  [4] sim=0.563 | P012774 (PAN, E2024): val/g=18.7, pts/g=20.1, ast/g=4.2, reb/g=3.5, stl/g=1.0, m...
  [5] sim=0.562 | P013405 (ASV, E2024): val/g=10.5, pts/g=8.4, ast/g=1.4, reb/g=6.2, stl/g=1.2, mi...
Step 2 — RAG prompt: 5 chunks
Step 3 — Generate (Ollama)...

Απαντηση [ollama]:
-----------------------------------------------------------------
Δεν βρηκα αυτη την πληροφορια.
-----------------------------------------------------------------


---
## 5. Φερε τα Δικα σου Δεδομενα

### Template για την εργασια

Αλλαζεις **μονο** τη `load_my_documents()` και το `DATASET='custom'` στα Knobs.  
Το υπολοιπο pipeline (embed, store, search, RAG) παραμενει ιδιο!

| Τυπος | Βιβλιοθηκη | Εγκατασταση |
|-------|-----------|-------------|
| TXT / Markdown | built-in | — |
| CSV | pandas | — |
| PDF | pypdf | `pip install pypdf` |
| DOCX | python-docx | `pip install python-docx` |
| Wikipedia | wikipedia-api | `pip install wikipedia-api` |

> **Στοχος εργασιας:** RAG chatbot σε δικα σου δεδομενα (≥20 docs η ≥500 γραμμες)

In [10]:
def load_my_documents():
    """
    Template — αντικατεστησε με τα δικα σου δεδομενα.
    Επιστρεφε: (corpus: list[str], ids: list[str], metas: list[dict])
    """

    # ── ΕΠΙΛΟΓΗ 1: TXT αρχεια ─────────────────────────────────────────────
    # import glob
    # corpus, ids, metas = [], [], []
    # for fpath in glob.glob('my_docs/*.txt'):
    #     with open(fpath, encoding='utf-8') as f:
    #         text = f.read()
    #     chunk_size, overlap = 500, 50
    #     for i in range(0, len(text), chunk_size - overlap):
    #         chunk = text[i:i+chunk_size].strip()
    #         if len(chunk) > 50:
    #             corpus.append(chunk)
    #             ids.append(f'{os.path.basename(fpath)}_{i}')
    #             metas.append({'source': fpath})
    # return corpus, ids, metas

    # ── ΕΠΙΛΟΓΗ 2: CSV ────────────────────────────────────────────────────
    # df = pd.read_csv('my_data.csv')
    # corpus = df['text_column'].dropna().tolist()
    # ids    = [f'row_{i}' for i in range(len(corpus))]
    # metas  = [{'row': i} for i in range(len(corpus))]
    # return corpus, ids, metas

    # ── ΕΠΙΛΟΓΗ 3: PDF (pip install pypdf) ───────────────────────────────
    # from pypdf import PdfReader
    # import glob
    # corpus, ids, metas = [], [], []
    # for fpath in glob.glob('my_pdfs/*.pdf'):
    #     reader = PdfReader(fpath)
    #     for pnum, page in enumerate(reader.pages):
    #         text = page.extract_text() or ''
    #         if len(text.strip()) > 50:
    #             corpus.append(text.strip())
    #             ids.append(f'{os.path.basename(fpath)}_p{pnum}')
    #             metas.append({'source': fpath, 'page': pnum})
    # return corpus, ids, metas

    # ── ΕΠΙΛΟΓΗ 4: Wikipedia (pip install wikipedia-api) ─────────────────
    # import wikipediaapi
    # wiki = wikipediaapi.Wikipedia('el')  # 'en' για αγγλικα
    # topics = ['Τεχνητη νοημοσυνη', 'Μηχανικη μαθηση']
    # corpus, ids, metas = [], [], []
    # for topic in topics:
    #     page = wiki.page(topic)
    #     if page.exists():
    #         for i, sec in enumerate(page.sections):
    #             if len(sec.text) > 50:
    #                 corpus.append(f'{topic} - {sec.title}: {sec.text[:500]}')
    #                 ids.append(f'{topic}_{i}')
    #                 metas.append({'topic': topic, 'section': sec.title})
    # return corpus, ids, metas

    # ── DEFAULT demo ─────────────────────────────────────────────────────
    demo = [
        'Το ΠΜΣ Εφαρμοσμενης Πληροφορικης διαρκει 2 χρονια και απαιτει 120 ECTS.',
        'Για το πτυχιο απαιτειται διπλωματικη εργασια 30 ECTS και παρουσιαση.',
        'Τα μαθηματα χωριζονται σε υποχρεωτικα (60 ECTS) και επιλογης (30 ECTS).',
        'Η δηλωση μαθηματων γινεται τον Σεπτεμβριο και τον Φεβρουαριο.',
        'Ο μεγιστος αριθμος φοιτητων ανα ετος ειναι 30.',
    ]
    return demo, [f'demo_{i}' for i in range(len(demo))], [{'source': 'demo'}]*len(demo)


if DATASET == 'custom':
    print('Φορτωση custom δεδομενων...')
    corpus_c, ids_c, metas_c = load_my_documents()
    if corpus_c:
        try: chroma_client.delete_collection(COLLECTION_NAME + '_custom')
        except: pass
        coll_c = chroma_client.create_collection(COLLECTION_NAME + '_custom',
                                                  metadata={'hnsw:space': 'cosine'})
        emb_c = embedder.encode(corpus_c, show_progress_bar=True)
        coll_c.add(documents=corpus_c, embeddings=emb_c.tolist(),
                   ids=ids_c, metadatas=metas_c)
        collection = coll_c   # override global
        corpus     = corpus_c
        print(f'Custom store ready: {collection.count()} chunks')
        rag_query('Τι χρειαζεται για το πτυχιο;')
    else:
        print('Corpus κενο — συμπληρωσε load_my_documents()!')
else:
    print("Αυτο το section τρεχει με DATASET='custom'.")
    print('Αλλαξε το knob και τρεξε ξανα!')

Αυτο το section τρεχει με DATASET='custom'.
Αλλαξε το knob και τρεξε ξανα!


---
## 6. Αξιολογηση Retrieval Quality

### Απλη ανθρωπινη αξιολογηση

Για καθε ερωτηση με γνωστη απαντηση, ελεγχουμε αν τα retrieved chunks  
περιεχουν τη σωστη πληροφορια (keyword check).

**Για πληρη αξιολογηση**: δοκιμασε το [RAGAS framework](https://ragas.io/) (advanced).

In [11]:
eval_qs = [
    {'q': 'Ποιος εχει τα περισσοτερα assists;', 'kw': 'ast'},
    {'q': 'Ποιος ειναι ο top scorer;',           'kw': 'pts'},
    {'q': 'Ποιος ειναι starter;',                'kw': 'starter'},
]

if collection.count() > 0:
    print('Αξιολογηση Retrieval Quality')
    print('=' * 55)
    scores = []
    for item in eval_qs:
        docs_r, sims_r, _ = semantic_search(item['q'], n=3)
        hit = any(item['kw'].lower() in d.lower() for d in docs_r)
        scores.append(int(hit))
        print(f"{'OK' if hit else 'MISS'} | sim={sims_r[0]:.3f} | Q: {item['q']}")
        print(f'  Best: {docs_r[0][:80]}...')
        print()
    print(f'Retrieval Precision: {sum(scores)}/{len(scores)} = {sum(scores)/len(scores)*100:.0f}%')
    print()
    print('Βελτιωση:')
    print('  - Αυξησε N_RESULTS')
    print('  - Δοκιμασε all-mpnet-base-v2 (καλυτερο embedding)')
    print('  - Προσθεσε περισσοτερα/καλυτερα chunks στο corpus')
else:
    print('Τρεξε πρωτα τα Sections 1-2.')

Αξιολογηση Retrieval Quality
OK | sim=0.612 | Q: Ποιος εχει τα περισσοτερα assists;
  Best: P007870 (PAR, E2024): val/g=10.8, pts/g=9.9, ast/g=1.5, reb/g=3.6, stl/g=0.9, mi...

OK | sim=0.619 | Q: Ποιος ειναι ο top scorer;
  Best: P007870 (PAR, E2024): val/g=10.8, pts/g=9.9, ast/g=1.5, reb/g=3.6, stl/g=0.9, mi...

OK | sim=0.587 | Q: Ποιος ειναι starter;
  Best: P008173 (BER, E2024): val/g=8.8, pts/g=9.6, ast/g=3.8, reb/g=2.0, stl/g=0.5, min...

Retrieval Precision: 3/3 = 100%

Βελτιωση:
  - Αυξησε N_RESULTS
  - Δοκιμασε all-mpnet-base-v2 (καλυτερο embedding)
  - Προσθεσε περισσοτερα/καλυτερα chunks στο corpus


---
## Ερωτησεις Αξιολογησης

Απαντα στις παρακατω ερωτησεις προσθετοντας **νεα markdown cells** μετα απο καθε ερωτηση.

---

**1. Out-of-corpus query:**  
Ρωτα κατι που σιγουρα ΔΕΝ υπαρχει στα docs (π.χ. 'Ποιος κερδισε το NBA Finals 2024;').  
Τι απανταει; Ποια chunks επεστρεψε; Τι συμπεραινεις;

---


1.Η ερώτηση που έκανα ήταν τελείως διαφορετική από το περιεχόμενο του csv αρχείου που φόρτωσα. Αυτό είχε σαν αποτέλεσμα το σύστημα αναζήτησης να επιστρέψει τα top 5(Εγώ αποφάσισα να είναι τόσα θα μπορούσε να ήταν και 10) εντελώς άσχετα chunks με στατιστικά βασικών παικτών που το similarity score ήταν αρκετά χαμηλό(0.48 - 0.49). Αυτό συνέβει διότι το RAG είναι αναγκασμένο να επιστρέφει πάντα κάποια αποτελέσματα(ανάλογα την επιλογή μας) ανεξαρτήτου ερώτησης με την μόνη διαφορά οτι αν η ερώτηση δεν είναι σχετική κάνει την μετατροπή σε διανύσματα φέρνοντας ουσιαστικά τα λιγότερο άσχετα διανύσματα που διαθέτει με βάσει τα διανύσματα της ερώτησης. Ωστόσο το μοντέλο (LLM) βλέπουμε οτι λειτουργεί ως φίλτρο καθώς διαβάζοντας τα chunks κατάλαβε ότι το domain αφορά στατιστικά μπάσκετ και όχι την Ιαπωνία που έγινε η ερώτηση. Αυτό είχε σαν αποτέλεσμα να τόνίσει αυτή τη διαφορά και να μην απαντήσει από το να επινοήσει μια απάντηση (hallucination) ή να προσπαθήσει να βρεί την λιγότερο άσχετη απάντηση. Με αυτό βλέπουμε πόσο σημαντικό έιναι ένα μοντέλο (LLM) σε ένα "κουτό" σύστημα αναζήτησης RAG. 

<img src="rag1.jpg" width="400">



**2. N_RESULTS tuning:**  
Αλλαξε N_RESULTS απο 3 → 7. Τρεξε ξανα μια ερωτηση.  
Πως αλλαζει η ποιοτητα; Γιατι 'περισσοτερα chunks' δεν σημαινει παντα 'καλυτερα';

--

2.Αλλάζοντας το Top-k chunks ουσιαστικά από 5 σε 7, παρατηρούμε ότι η ποιότητα της απάντησης δεν θα βελτιωθεί απαραίτητα διότι ο κανόνας που λέει οτι περισσότερα chunks σημαίνει και καλύτερα ή μάλλον πιο σωστά αποτελέσματα δεν ισχύει στα συστήματα RAG. Αφού του είπαμε να δώσει περισσότερα αποτελέσματα ουσιαστικά λέμε στο το σύστημα να αναζητήσει και να επιστρέψει διανύσματα με χαμηλότερο similarity score μόνο και μόνο για να καλύψει τις θέσεις που του ζητήσαμε στην αρχή εισάγοντας έτσι λανθασμένη πληροφορία. Αυτή η λάθος πληροφορία μπερδεύει το μοντέλο (LLM) το οποίο καλείται να εντοπίσει τη σωστή πληροφορία μέσα σε ένα πολύ μεγαλύτερο και κείμενο, αυξάνοντας τον κίνδυνο για hallucinations τις καθυστερήσεις στην απόκριση και την κατανάλωση (RAM).


<img src="rag2.jpg" width="400">

-

**3. Retrieval vs Generation:**  
Εξηγησε με δικα σου λογια (3-5 προτασεις) γιατι το RAG δινει καλυτερες απαντησεις  
απο plain LLM για domain-specific ερωτησεις.

--

3.Ένα απλό LLM ουσιαστικά βασίζεται αποκλειστικά στα δεδομένα που του δώσαμε κατά το στάδιο της εκπαίδευσης του και να τα μάθει από αυτά με αποτέλεσμα να μην έχει πρόσβαση σε πιο to the point-πρόσφατες ή domain πληροφορίες κάνοντας πολλές φορές hallucinations όταν δεν γνωρίζει την απάντηση μόνο και μόνο για να απαντήσει. Το σύστημα RAG από την άλλη πάει ένα βήμα παραπάνω αυτό της αναζήτησης δηλαδή πρώτα βρίσκει τα ακριβή και δεδομένα από τη πηγή που το έχουμε δώσει και στη συνέχεια τα παιρναέι στο LLM έτσι το μοντέλο σταματά να μαντεύει και λειτουργεί ως ένας βοηθός που δίνει απαντήσεις βασισμένος αυστηρά σε πραγματικά δεδομένα που εμείς του δώσαμε με την προυπόθεση πως πρέπει οι ερώτησεις που θα γίνουν να είναι βάσει αυτών των δεδομένων.

-

**4. Embedding model:**  
Αλλαξε EMBED_MODEL σε 'all-mpnet-base-v2'.  
Παρατηρησε χρονο + retrieval scores. Ποιο προτιμας;

--

4. Αλλάζοντας το μοντέλο σε all-mpnet-base-v2, παρατήρησα δύο βασικά πράγματα. Πρώτον, ο χρόνος φόρτωσης και αναζήτησης ήταν σχεδόν ίδιος μάλλον λόγω του μικρού μεγέθους του dataset. Δεύτερον τα similarity scores αυξήθηκαν αισθητά όπως φαίνεται γιατί άγγιξαν το 0.68 σε σχέση του 0.62 του προηγούμενου μοντέλου. Το νέο μοντέλο είναι πιο ισχυρό και αποτυπώνει καλύτερα τη νοηματική συνάφεια των λέξεων. Τέλος το παράδειγμα τόνισε το πρόβλημα των δεδομένων καθώς παρόλοπου βρέθηκε το υψηλότερο σκορ (pts/g=19.8) το LLM δεν μπόρεσε να κατονομάσει τον παίκτη διότι τα chunks περιέχουν μόνο κωδικούς (π.χ. P003469) και όχι ονόματα. Αυτό επιβεβαιώνει τη χρησιμότητα του LLM ως φίλτρο που απαντήθηκε στο ερώτημα 3. προτιμώ το νέο μοντέλο, διότι RAG προσφέρει μεγαλύτερη ακρίβεια.

<img src="rag4.jpg" width="400">

-

**5. [Εργασια] Σχεδιασμος:**  
- Τι δεδομενα θα χρησιμοποιησεις (πηγη, μεγεθος, μορφη)
- 5 sample ερωτησεις που θελεις να απανταει το chatbot
- Ποιο chunking strategy επιλεγεις και γιατι

**5. Θέμα:** Chatbot Γεωστατιστικής» για ανάλυση αποθεμάτων λιθίου. (Έμπνευση από την διπλωματική)
* **Πηγή:** Ανοιχτά δεδομένα Kaggle αρχείο (lithium.csv).
* **Μέγεθος:** 24 (chunks).
* **Μορφή:** Τα αρχεία θα είναι τύπου .pdf και .csv
* **Ερωτήσεις που θα κάνω:**
  1. Which country has the highest lithium reserves?
  2. What are the reported reserves for Australia and Chile?
  3. List the top countries with the largest lithium deposits.
  4. Which country ranks first in lithium production?
  5. Provide the statistical profile for a specific country.
* **Chunking Strategy:** Row-based chunking (κάθε ολόκληρη γραμμή του CSV αποτελεί ένα chunk) γιατί εξασφαλίζει ότι οι αριθμοί και τα στατιστικά κάθε χώρας δεν πειράζονται και το LLM δεν χάνει το νόημά τους (πχ αν τα δεδομένα κόβονταν στη μέση ανά χαρακτήρα).

<img src="rag5.jpg" width="400">      